# Chapter 14: RAG Evaluation End-to-End
## Topic 4: Tracing and Observability — Arize Phoenix and Custom Logging

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- Every earlier topic in this notebook series that recommended "logging" as a debugging or monitoring aid — Chapter 7 Topic 8's filtering logs, Chapter 8 Topic 2's citation logs, Chapter 9 Topic 7's segmented hallucination monitoring, Chapter 10 Topic 1's tool-call request/result logging — has treated logging as an ad hoc, per-component concern, each piece logged separately in whatever way that specific topic's discussion happened to suggest. This topic asks the natural next question: what does it look like to unify all of this into one coherent, structured observability practice, rather than a patchwork of disconnected log statements scattered across the pipeline?
- **Tracing**, specifically, is the discipline of recording a single request's complete journey through every stage of a multi-step pipeline — retrieval, reranking, generation, tool calls — as one connected structure (a "trace" made up of individual "spans," one per stage), rather than as isolated log lines with no explicit connection between them. This directly solves the layered-attribution problem this notebook series has repeatedly required solving manually: instead of separately checking retrieval logs, then generation logs, then tool-call logs to reconstruct what happened for one specific request, a trace presents that entire journey as a single, navigable object.
- **Arize Phoenix**, specifically, is an open-source observability tool built for exactly this kind of LLM-pipeline tracing — it provides a UI for visualizing traces, and integrates with the OpenTelemetry standard (a broader, industry-standard tracing protocol, not unique to LLM applications) for actually recording and transmitting trace data. This connects directly to Chapter 11 Topic 3's MCP discussion: both MCP and OpenTelemetry are examples of adopting a standardized protocol rather than building a fully bespoke solution, and the same evidence-based adoption discipline (does the standardization benefit outweigh the added complexity at this project's actual scale) applies to evaluating Phoenix adoption too.


### 2. Internal Working — Step by Step

**How tracing actually works, mechanically, using OpenTelemetry's real concepts:**

1. **A "trace" represents one complete request's journey** — for this project, one email's full path through classification, retrieval, and generation. A trace has a unique ID, and every stage's recorded activity for that request gets tagged with this same trace ID, which is what makes it possible to later reconstruct the complete, connected journey.
2. **A "span" represents one specific stage or operation within that trace** — a retrieval call, a tool invocation, a generation call — each with its own start time, end time (giving real, measured latency per stage, directly relevant to Chapter 8 Topic 1's token-budgeting and Chapter 10 Topic 2's message-list-growth cost concerns), and arbitrary structured attributes (the query text, the retrieved chunk count, the tool name called).
3. **Spans can be nested**, reflecting the actual structure of a multi-step pipeline — a top-level span for "handle this email" might contain child spans for "retrieval," "reranking," and "generation," each of which might itself contain further child spans (a specific tool call within generation, for example) — this nested structure is what directly replaces the ad hoc, disconnected log statements this notebook series has used throughout, with one coherent, hierarchical record.
4. **Phoenix's specific contribution is visualizing this trace/span structure in a UI**, letting a developer inspect exactly which stage of a specific request took how long, what it received as input, and what it produced as output — directly, visually, rather than manually cross-referencing separate log files or print statements the way this notebook series' earlier ad hoc logging recommendations would require.


### 3. How This Is Implemented in This Project

- Every one of this project's pipeline stages already discussed logging in isolation — this topic's contribution is wrapping each of those stages (retrieval, Chapter 7; tool calls, Chapter 10; generation, Chapter 8) in explicit tracing spans, using OpenTelemetry's real API, so that a single email's complete journey through all of them becomes one connected, inspectable trace rather than separate, disconnected log statements.
- This notebook demonstrates real, working OpenTelemetry span creation and Phoenix's actual trace-recording mechanism (using the genuine, installed `arize-phoenix` and `opentelemetry` packages) — not a simulation — applied to this project's actual retrieval-and-generation flow, producing real trace data that Phoenix's tools can process, even though this notebook environment can't launch Phoenix's full interactive web UI for visual inspection.
- Following Chapter 11 Topic 3's own adoption-trigger discipline, adopting a full observability platform like Phoenix (rather than continuing with this notebook series' existing pattern of targeted, per-component logging) is worth evaluating once this project's pipeline complexity (Chapter 10 Topic 6's multi-tool agents, Chapter 13's agentic RAG) grows enough that manually cross-referencing separate logs becomes genuinely difficult — not adopted simply because a more sophisticated tool exists.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **Tracing itself has a real, if usually small, performance overhead** — recording span start/end times and attributes for every pipeline stage adds a small amount of processing per request, generally negligible compared to the cost of an actual LLM call or retrieval operation, but worth being aware of rather than assuming observability instrumentation is entirely free.
- **A trace is only as useful as the attributes actually recorded on each span** — a span that records only "retrieval happened" without also recording the actual query, the number of results, and their scores provides far less debugging value than one recording this project's actual, specific data at each stage — exactly the same "log enough detail to actually be useful" principle Chapter 7 Topic 8 and Chapter 8 Topic 2 already established for their own specific logging recommendations, now applied uniformly across every stage via a consistent tracing structure.
- **Sensitive data (customer PII, actual email content) flowing through trace attributes needs the same security consideration as any other logged data** — Chapter 9 Topic 4's customer-record access-control principles apply directly to trace data too, since a trace capturing a customer's email content and any looked-up account details is, in effect, another place that sensitive data now lives and needs to be protected and access-controlled.
- **Debugging with tracing genuinely changes the debugging workflow this notebook series has used throughout** — rather than manually reconstructing what happened by checking several separate logs (Chapter 10 Topic 1's manual three-layer debugging discipline, for example), a single trace view directly shows the connected sequence of spans for one specific request, collapsing what used to require cross-referencing multiple sources into one navigable structure.
- **Monitoring at scale:** once tracing is in place, aggregating span data across many traces (average retrieval latency, tool-call frequency distribution, error rates per span type) becomes possible in a way ad hoc, per-component logging doesn't naturally support — directly enabling more sophisticated versions of every monitoring recommendation this notebook series has already made (Chapter 9 Topic 7's hallucination-rate segmentation, Chapter 10 Topic 6's tool-call-combination tracking) as queries over structured trace data rather than manual log inspection.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **Adopting a full observability platform (Phoenix) vs continuing this notebook series' existing pattern of targeted, per-component logging:** a full platform provides a unified, queryable, visualizable structure across the entire pipeline, at the cost of a new dependency and genuine setup/maintenance effort — worth adopting once pipeline complexity genuinely makes manual log cross-referencing difficult, mirroring exactly the same evidence-based adoption discipline already applied to MCP (Chapter 11 Topic 3) and the vector database migration (Chapter 6).
- **How much detail to record in span attributes:** recording more detail (full query text, full retrieved chunk content, full generated answer) maximizes debugging value but increases both the security-sensitivity of the trace data itself and its storage cost at scale — a genuine trade-off requiring the same care already established for what to log in Chapter 7 Topic 8 and Chapter 8 Topic 2's specific recommendations, now needing a consistent, project-wide policy rather than being decided independently per component.
- **Whether to trace every single request or a sampled subset:** tracing every request maximizes debugging coverage but incurs the most overhead and storage cost at real production volume (Chapter 19's future latency/cost concerns); sampling a representative subset reduces this cost while still providing meaningful aggregate and example-level visibility — a similar sampling trade-off to Chapter 9 Topic 7's hallucination-rate monitoring sampling strategy.


### 6. Alternatives and When to Use Each

- **Ad hoc, per-component logging (this notebook series' existing default pattern):** appropriate for a project at a scale where manually cross-referencing a handful of separate logs remains tractable, and where the cost of adopting a new observability platform isn't yet justified.
- **A full tracing/observability platform (Arize Phoenix, or an equivalent):** worth adopting once pipeline complexity (many tools, multi-turn conversations, agentic RAG's variable retrieval-retry behavior) makes manual log cross-referencing genuinely difficult, providing a unified, structured, queryable view across the entire pipeline.
- **Custom, in-house structured logging (recording the same trace/span-style structured data without adopting a full external platform):** a middle-ground option, capturing some of tracing's structural benefit (connected, hierarchical records) without a new external dependency — worth considering if Phoenix's specific additional features (its UI, its LLM-specific instrumentation helpers) aren't judged worth the dependency cost, but the underlying structured-tracing benefit still is.


### 7. Common Mistakes and Production Failures

- Continuing to rely on scattered, ad hoc, per-component logging well past the point where pipeline complexity has made manually cross-referencing them genuinely difficult, rather than recognizing this as the concrete trigger for adopting structured tracing.
- Recording spans with insufficient attribute detail, reproducing the "log exists but isn't actually useful for debugging" problem this notebook series has repeatedly warned against for individual logging recommendations, now at the scale of an entire tracing system.
- Not applying the same security and access-control discipline to trace data (which can contain customer PII and account details) as to any other sensitive data store in the project.
- Adopting a full observability platform preemptively, before pipeline complexity genuinely justifies it, incurring real dependency and maintenance cost without a corresponding, measured need — the same premature-adoption risk already flagged for MCP in Chapter 11 Topic 3.
- Tracing every single request at full production volume without considering the real overhead and storage cost this incurs, rather than a deliberate, cost-conscious sampling strategy.


### 8. Lead-Level Interview Questions

**Basic**

- Q: What's the difference between a "trace" and a "span" in observability terminology?
  A: A trace represents one complete request's entire journey through a system — for this project, one email's full path through classification, retrieval, and generation. A span represents one specific stage or operation within that trace — a single retrieval call, a single tool invocation — each with its own timing and attributes. A trace is composed of multiple, often nested, spans.

- Q: How does tracing solve the layered-attribution problem this notebook series has repeatedly required solving manually?
  A: Instead of separately checking retrieval logs, generation logs, and tool-call logs to reconstruct what happened for one specific request — the manual, multi-source debugging approach used throughout this notebook series — a trace presents that entire connected journey as a single, navigable structure, directly showing which specific stage did what, in what order, with what timing.

**Intermediate**

- Q: Why does adopting a full observability platform like Arize Phoenix require the same evidence-based adoption discipline already applied to MCP in Chapter 11 Topic 3?
  A: Both are standardized-protocol adoptions with real trade-offs — added dependency and setup/maintenance cost in exchange for structural benefits (unified tool access for MCP, unified request visibility for tracing) that only pay off once genuine complexity justifies them. Adopting either preemptively, before that complexity exists, incurs real cost without a corresponding measured benefit, exactly the premature-adoption risk this notebook series has repeatedly warned against.

- Q: Why does a span's recorded attribute detail matter as much as the fact that tracing exists at all?
  A: A span recording only that an operation happened, without capturing the actual query, retrieved results, or relevant parameters, provides little more debugging value than no tracing at all — exactly the same "log enough detail to actually be useful" principle already established for specific logging recommendations elsewhere in this notebook series (Chapter 7 Topic 8, Chapter 8 Topic 2), now needing to be applied consistently across every traced stage.

**Advanced**

- Q: Design a tracing strategy for this project's actual multi-tool agent (Chapter 10 Topic 6) and agentic RAG (Chapter 13) pipeline, specifically addressing what should be captured at each stage.
  A: A top-level span for the full email-handling request, with child spans for each major stage: rule-based classification (Chapter 1), the agent loop's tool-selection decisions (capturing which tools were considered and which were actually called, directly enabling Chapter 10 Topic 7's tool-selection-accuracy monitoring), each individual tool call (capturing the specific arguments and results, mirroring Chapter 10 Topic 1's request/dispatch/result logging), and the final generation step (capturing the retrieved context and generated answer, enabling Chapter 8's faithfulness checks to be run against the same trace data later). This structure directly supports reconstructing and debugging any specific request's complete journey without needing to manually cross-reference separate logs.

- Q: A teammate proposes tracing every single production request in full detail once this project reaches its stated 8,000-12,000 emails/day volume. What would you push back on?
  A: This introduces real overhead and storage cost at that scale, and full-detail tracing of every request may not be necessary if the goal is representative visibility and aggregate monitoring rather than guaranteed per-request debugging capability. A sampled tracing strategy — full detail on a representative subset, lighter-weight aggregate metrics on the rest — mirrors Chapter 9 Topic 7's own hallucination-rate sampling discipline, and would likely capture most of the practical debugging and monitoring value at meaningfully lower cost.

**Scenario-based**

- Q: A specific customer reports a confusing, seemingly incorrect response, and you have full tracing in place. Walk through how you'd use it to debug this, compared to this project's earlier, ad hoc logging approach.
  A: With tracing, you'd look up the specific trace ID for that customer's request (if request IDs are correlated with trace IDs, a natural design choice) and inspect its complete, connected span sequence directly — seeing exactly what the rule-based classifier decided, which tools the agent chose to call and with what arguments, what each tool returned, and what context the generation step received and produced, all in one navigable view. This replaces what would otherwise require manually finding and cross-referencing the relevant entries across several separate logs (Chapter 10 Topic 1's classification log, Chapter 7's retrieval log, Chapter 8's generation log) for the same request — a genuinely faster, less error-prone debugging process once tracing is properly in place.

**Follow-up questions to expect:**

- "How would you decide what sampling rate is appropriate for production tracing at this project's stated volume?"
- "What security review would you want before storing customer email content in trace attributes?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **OpenTelemetry, the underlying protocol Phoenix builds on, is a broader industry standard for tracing and observability, not unique to LLM applications** — recognizing tracing as an instance of this general, widely-adopted software observability practice (rather than something invented specifically for LLM pipelines) connects this topic to a much broader body of applicable software engineering practice.
- **The trace/span hierarchical structure is a specific instance of a general pattern for representing nested, multi-step processes** — the same nested-structure idea recurs anywhere a complex operation needs to be broken down into inspectable sub-operations, well beyond observability specifically.
- **This topic's tracing discipline is the natural infrastructure underlying Topic 5's A/B testing**: comparing two retrieval strategies' actual behavior in detail (not just their aggregate RAGAS scores) benefits directly from having traced, inspectable records of exactly what each strategy actually did for specific queries, connecting this topic's infrastructure to the next topic's comparison methodology.

### 10. Quick Revision Summary (for last-minute interview prep)

> Tracing unifies this notebook series' many separate, ad hoc logging recommendations (Chapter 7 Topic 8's filtering logs, Chapter 8 Topic 2's citation logs, Chapter 10 Topic 1's tool-call logs) into one coherent structure: a trace represents one complete request's full journey, composed of nested spans, each representing one specific pipeline stage with its own timing and structured attributes. This directly solves the layered-attribution problem this notebook series has repeatedly required solving manually — instead of cross-referencing separate logs, a single trace presents the complete, connected sequence of what happened for one specific request. Arize Phoenix, built on the OpenTelemetry industry standard, provides tooling specifically for recording and visualizing this trace/span structure for LLM pipelines. Like MCP (Chapter 11 Topic 3), adopting a full observability platform is a genuine architectural decision with real trade-offs — dependency and maintenance cost in exchange for unified visibility — that should follow from a concrete complexity trigger (many tools, agentic retrieval-retry behavior, multi-turn conversations making manual log cross-referencing genuinely difficult), not be adopted simply because a more sophisticated tool exists. Trace data, since it can contain customer PII and account details, needs the same security and access-control discipline as any other sensitive data store in this project.


### Module 1: Real OpenTelemetry Tracer Setup — Verified via In-Memory Export

Set up a REAL OpenTelemetry tracer (the same underlying technology Arize Phoenix uses), with an in-memory exporter so we can inspect actual, real span data programmatically, since this environment can't display Phoenix's interactive web UI.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: Real OpenTelemetry tracer setup
#
# We use the REAL, installed opentelemetry SDK -- the same underlying
# technology Arize Phoenix is built on. We use an in-memory exporter
# (rather than actually launching Phoenix's web server) so we can
# inspect REAL, genuine span data programmatically in this notebook
# environment, which cannot display an interactive web UI.
# ------------------------------------------------------------------

from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

# REAL tracer provider setup -- exactly what a real Phoenix integration
# would configure, just pointed at an in-memory exporter for inspection
exporter = InMemorySpanExporter()
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(exporter))
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("fd-email-pipeline")

print("=" * 70)
print("MODULE 1: REAL OPENTELEMETRY TRACER CONFIGURED")
print("=" * 70)
print("This is the REAL opentelemetry SDK -- the same underlying")
print("technology arize-phoenix uses for tracing. We use an in-memory")
print("exporter here so we can inspect actual span data directly,")
print("since this environment cannot display Phoenix's web UI.")

# a quick, real test span to confirm the setup works
with tracer.start_as_current_span("test_span") as span:
    span.set_attribute("test_attribute", "hello_tracing")

captured_spans = exporter.get_finished_spans()
print(f"\nTest span captured: {len(captured_spans)} span(s)")
print(f"  Span name: {captured_spans[0].name}")
print(f"  Span attributes: {dict(captured_spans[0].attributes)}")

exporter.clear()
print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: REAL OPENTELEMETRY TRACER CONFIGURED
This is the REAL opentelemetry SDK -- the same underlying
technology arize-phoenix uses for tracing. We use an in-memory
exporter here so we can inspect actual span data directly,
since this environment cannot display Phoenix's web UI.

Test span captured: 1 span(s)
  Span name: test_span
  Span attributes: {'test_attribute': 'hello_tracing'}

Module 1 complete. Run Module 2 next.


### Module 2: Tracing This Project's Real Pipeline — Nested Spans

Wrap this project's actual retrieval, tool-call, and generation steps in REAL, nested OpenTelemetry spans, exactly the trace/span structure the theory describes.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: Real, nested spans across this project's actual pipeline
# ------------------------------------------------------------------

from rank_bm25 import BM25Okapi

def tokenize(text: str) -> list:
    return text.lower().split()

KNOWLEDGE_BASE = [
    "Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate.",
    "Senior citizens receive an additional 0.5 percent interest on all Fixed Deposit tenures.",
]
tokenized_corpus = [tokenize(doc) for doc in KNOWLEDGE_BASE]
bm25 = BM25Okapi(tokenized_corpus)

FD_RECORDS = {"BJ2019FD7717": {"customer_name": "Shobha Chopra", "status": "Closed (Premature)"}}


def traced_retrieve(query: str) -> list:
    """Retrieval wrapped in a REAL span, with REAL attributes recorded --
    exactly the 'log enough detail to be useful' principle applied
    consistently via tracing."""
    with tracer.start_as_current_span("retrieval") as span:
        span.set_attribute("query", query)
        scores = bm25.get_scores(tokenize(query))
        ranked = sorted(range(len(KNOWLEDGE_BASE)), key=lambda i: scores[i], reverse=True)
        results = [KNOWLEDGE_BASE[i] for i in ranked[:1]]
        span.set_attribute("num_results", len(results))
        span.set_attribute("top_score", float(scores[ranked[0]]))
        return results


def traced_tool_call(reference_number: str) -> dict:
    """Tool call wrapped in a REAL span -- mirrors Chapter 10 Topic 1's
    request/dispatch/result logging, now as a structured span."""
    with tracer.start_as_current_span("tool_call.validate_fd_reference") as span:
        span.set_attribute("reference_number", reference_number)
        record = FD_RECORDS.get(reference_number)
        found = record is not None
        span.set_attribute("found", found)
        return {"found": found, **(record or {})}


def traced_generate(query: str, context: list) -> str:
    """Generation wrapped in a REAL span."""
    with tracer.start_as_current_span("generation") as span:
        span.set_attribute("context_chunk_count", len(context))
        answer = context[0] if context else "insufficient context"
        span.set_attribute("answer_length", len(answer))
        return answer


def handle_email_traced(email_content: str, reference_number: str = None) -> dict:
    """The TOP-LEVEL span for one complete email-handling request --
    contains NESTED child spans for each pipeline stage, exactly the
    hierarchical trace/span structure this topic's theory describes."""
    with tracer.start_as_current_span("handle_email") as root_span:
        root_span.set_attribute("email_preview", email_content[:50])

        tool_result = None
        if reference_number:
            tool_result = traced_tool_call(reference_number)

        retrieved = traced_retrieve(email_content)
        answer = traced_generate(email_content, retrieved)

        root_span.set_attribute("final_answer_preview", answer[:50])
        return {"answer": answer, "tool_result": tool_result}


print("=" * 70)
print("MODULE 2: TRACING A REAL EMAIL THROUGH THE FULL PIPELINE")
print("=" * 70)

result = handle_email_traced(
    "What is the penalty for premature FD withdrawal on account BJ2019FD7717?",
    reference_number="BJ2019FD7717"
)
print(f"\nFinal result: {result}")

captured_spans = exporter.get_finished_spans()
print(f"\n{len(captured_spans)} REAL spans captured for this ONE email-handling request:")
for span in captured_spans:
    print(f"  - {span.name} (attributes: {dict(span.attributes)})")

exporter.clear()
print("\nModule 2 complete. Run Module 3 next.")


MODULE 2: TRACING A REAL EMAIL THROUGH THE FULL PIPELINE

Final result: {'answer': 'Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate.', 'tool_result': {'found': True, 'customer_name': 'Shobha Chopra', 'status': 'Closed (Premature)'}}

4 REAL spans captured for this ONE email-handling request:
  - tool_call.validate_fd_reference (attributes: {'reference_number': 'BJ2019FD7717', 'found': True})
  - retrieval (attributes: {'query': 'What is the penalty for premature FD withdrawal on account BJ2019FD7717?', 'num_results': 1, 'top_score': -0.04947042763629408})
  - generation (attributes: {'context_chunk_count': 1, 'answer_length': 86})
  - handle_email (attributes: {'email_preview': 'What is the penalty for premature FD withdrawal on', 'final_answer_preview': 'Premature withdrawal of FD incurs a 1 percent pena'})

Module 2 complete. Run Module 3 next.


### Module 3: Reconstructing the Complete Trace — The Layered-Attribution Problem, Solved

Demonstrate the actual debugging value: given a trace ID, reconstruct the COMPLETE, connected journey of a specific request from real span data, without cross-referencing separate logs.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: Reconstructing a complete trace from real span data
# ------------------------------------------------------------------

# run TWO different emails through the traced pipeline
handle_email_traced("What is the penalty for premature withdrawal?")
handle_email_traced("Do senior citizens get extra interest?", reference_number="BJ2019FD7717")

all_spans = exporter.get_finished_spans()
print("=" * 70)
print("MODULE 3: RECONSTRUCTING COMPLETE TRACES FROM REAL SPAN DATA")
print("=" * 70)
print(f"\nTotal spans captured across 2 email-handling requests: {len(all_spans)}")

# group spans by their REAL trace ID -- this is the actual mechanism
# that lets us reconstruct one request's complete, connected journey
traces_by_id = {}
for span in all_spans:
    trace_id = span.context.trace_id
    traces_by_id.setdefault(trace_id, []).append(span)

print(f"Distinct traces (one per email-handling request): {len(traces_by_id)}")

for trace_id, spans in traces_by_id.items():
    # sort by start time to reconstruct the actual sequence of events
    spans_sorted = sorted(spans, key=lambda s: s.start_time)
    root_span = next(s for s in spans_sorted if s.name == "handle_email")
    email_preview = root_span.attributes.get("email_preview")

    print(f"\n--- TRACE for email: '{email_preview}...' ---")
    for span in spans_sorted:
        duration_ns = span.end_time - span.start_time
        print(f"  [{span.name}] duration={duration_ns/1000:.1f}us | attrs={dict(span.attributes)}")

print("\nCONFIRMED: each trace shows the COMPLETE, connected sequence of")
print("what happened for ONE specific email -- retrieval, tool calls, and")
print("generation, all reconstructable from real span data grouped by")
print("trace ID -- exactly the layered-attribution capability this")
print("notebook series has repeatedly required doing MANUALLY across")
print("separate logs, now available as one structured, queryable view.")

print("\nModule 3 complete. All Chapter 14 Topic 4 modules done.")
print()
print("=" * 70)
print("CHAPTER 14 TOPIC 4 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  Used the REAL, installed opentelemetry SDK (the same technology
  arize-phoenix builds on) -- not a simulation. Real TracerProvider,
  real spans, real trace IDs, verified via real in-memory export.

  A TRACE (one complete request) is composed of NESTED SPANS (one per
  pipeline stage) -- demonstrated concretely: one email produced 4 real,
  connected spans (handle_email -> tool_call -> retrieval -> generation).

  Grouping spans by trace ID reconstructs a request's COMPLETE journey
  -- demonstrated with 2 separate emails producing 2 cleanly separated
  traces, each independently reconstructable.

  This directly replaces the manual, cross-log debugging discipline
  this notebook series has used throughout (Chapter 10 Topic 1's
  three-layer debugging) with one structured, queryable view.

  Span attribute detail matters as much as tracing's existence --
  each span here recorded real, useful data (query, scores, results),
  not just "this happened."
""")


MODULE 3: RECONSTRUCTING COMPLETE TRACES FROM REAL SPAN DATA

Total spans captured across 2 email-handling requests: 7
Distinct traces (one per email-handling request): 2

--- TRACE for email: 'What is the penalty for premature withdrawal?...' ---
  [handle_email] duration=241.4us | attrs={'email_preview': 'What is the penalty for premature withdrawal?', 'final_answer_preview': 'Premature withdrawal of FD incurs a 1 percent pena'}
  [retrieval] duration=141.2us | attrs={'query': 'What is the penalty for premature withdrawal?', 'num_results': 1, 'top_score': 0.0}
  [generation] duration=10.0us | attrs={'context_chunk_count': 1, 'answer_length': 86}

--- TRACE for email: 'Do senior citizens get extra interest?...' ---
  [handle_email] duration=226.4us | attrs={'email_preview': 'Do senior citizens get extra interest?', 'final_answer_preview': 'Premature withdrawal of FD incurs a 1 percent pena'}
  [tool_call.validate_fd_reference] duration=99.2us | attrs={'reference_number': 'BJ2019FD7717